 # Librerias

 

In [13]:
import pandas as pd
import ast
import re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

 # Carga de datos 


In [2]:
dfentero = pd.read_csv(r"..\..\Data\clean_data_24-03-2026.csv",parse_dates=['insert_date','first_review_date','last_review_date'])

In [3]:
dfentero.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9650 entries, 0 to 9649
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 9650 non-null   int64         
 1   name                         9647 non-null   object        
 2   description                  9650 non-null   object        
 3   host_id                      9650 non-null   int64         
 4   neighbourhood_name           9650 non-null   object        
 5   neighbourhood_district       9650 non-null   object        
 6   room_type                    9650 non-null   object        
 7   accommodates                 9650 non-null   int64         
 8   bathrooms                    9576 non-null   float64       
 9   bedrooms                     9581 non-null   float64       
 10  beds                         9605 non-null   float64       
 11  amenities_list               9650 non-null 

In [4]:
df=dfentero[['apartment_id','neighbourhood_name','minimum_nights','maximum_nights','review_scores_rating','review_scores_location','city']].copy()

In [5]:
df.head(5)

,apartment_id,neighbourhood_name,minimum_nights,maximum_nights,review_scores_rating,review_scores_location,city
0,11964,Centro,3,365,97.0,100.0,Malaga
1,21853,C�rmenes,4,40,92.0,80.0,Madrid
2,32347,San Vicente,2,120,98.0,100.0,Sevilla
3,35379,l'Antiga Esquerra de l'Eixample,2,730,94.0,100.0,Barcelona
4,35801,Quart,1,180,97.0,100.0,Girona


In [6]:
listabarrios=df['neighbourhood_name'].unique().tolist()

In [7]:
listabarrios

['Centro',
 'C�rmenes',
 'San Vicente',
 "l'Antiga Esquerra de l'Eixample",
 'Quart',
 'Torroella de Fluvi�',
 "el Camp de l'Arpa del Clot",
 "la Dreta de l'Eixample",
 'Embajadores',
 "el Camp d'en Grassot i Gr�cia Nova",
 'el Raval',
 'el Fort Pienc',
 'Palacio',
 'Palomeras Bajas',
 'Lloret de Mar',
 'EL PILAR',
 'Vallvidrera, el Tibidabo i les Planes',
 'Sants',
 'Sant Antoni',
 'Forallac',
 'les Corts',
 'Palma de Mallorca',
 'Universidad',
 'Alc�dia',
 'Justicia',
 'Horta',
 'Sant Pere, Santa Caterina i la Ribera',
 'el Poble Sec',
 'EN CORTS',
 'Capmany',
 'Aluche',
 "Castell� d'Emp�ries",
 'Can Peguera',
 'Cortes',
 'Ciudad Jard�n',
 'la Vila de Gr�cia',
 "la Nova Esquerra de l'Eixample",
 'Ni�o Jes�s',
 'Sol',
 'RUSSAFA',
 'Vilapicina i la Torre Llobeta',
 'Arenal',
 'Begur',
 'Selva',
 'Alfalfa',
 'la Sagrada Fam�lia',
 'Santa Margalida',
 'EL CARME',
 'Tossa de Mar',
 'el Putxet i el Farr�',
 'el Barri G�tic',
 'Es Mercadal',
 'el Poblenou',
 'Campos',
 'S�ller',
 'Ciutadell

In [8]:
# 1. Pasamos todo a minúsculas y quitamos los espacios de los extremos
df['neighbourhood_name'] = df['neighbourhood_name'].str.lower().str.strip()



In [9]:
df['neighbourhood_name'].value_counts()

neighbourhood_name
centro                    325
la dreta de l'eixample    297
embajadores               278
palma de mallorca         209
el raval                  206
                         ... 
riudarenes                  1
ripoll                      1
setcases                    1
peralada                    1
darnius                     1
Name: count, Length: 520, dtype: int64

In [10]:
# Ambas condiciones tienen que cumplirse para que sea True
df['es_turistico'] = (df['minimum_nights'] <= 30) & (df['maximum_nights'] <= 30)

In [11]:
df['es_turistico'].value_counts()

es_turistico
False    7789
True     1861
Name: count, dtype: int64

In [16]:


# 1. SELECCIÓN DE VARIABLES Y LIMPIEZA DE NULOS
# Scikit-learn dará un error fatal si hay valores vacíos (NaN) en estas columnas. 
# Los rellenamos con la mediana del mercado para no alterar el modelo.
features = ['review_scores_rating', 'review_scores_location', 'minimum_nights', 'maximum_nights']
df_ml = df.copy()

for col in features:
    df_ml[col] = df_ml[col].fillna(df_ml[col].median())

X = df_ml[features]

# 2. PREPROCESAMIENTO: ESCALADO (MinMaxScaler)
# Aplastamos las métricas para que todas valgan entre 0 y 1.
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# 3. CREACIÓN DEL "PISO SINTÉTICO IDEAL"
# Al usar pd.DataFrame le pasamos los nombres de las columnas para que scikit-learn no se queje
piso_ideal = pd.DataFrame([[100, 100, 2, 30]], columns=features)

# Transformamos este piso ficticio usando el mismo escalador que los datos reales
piso_ideal_scaled = scaler.transform(piso_ideal)

# 4. ENTRENAMIENTO DEL MODELO DE MACHINE LEARNING
# Usamos NearestNeighbors para mapear todos los apartamentos en un espacio vectorial
modelo_nn = NearestNeighbors(metric='euclidean')
modelo_nn.fit(X_scaled) # Aquí es donde el modelo "aprende" la distribución

# 5. CÁLCULO DE DISTANCIAS AL PUNTO IDEAL
# Le pedimos al modelo la distancia euclidiana desde el piso ideal a TODOS los pisos reales
distancias, indices = modelo_nn.kneighbors(piso_ideal_scaled, n_neighbors=len(X_scaled))

# 6. ASIGNACIÓN DE RESULTADOS AL DATAFRAME
# Las distancias vienen ordenadas, así que las emparejamos con su índice correcto
serie_distancias = pd.Series(distancias[0], index=df_ml.index[indices[0]])
df_ml['Score_Optimizacion'] = serie_distancias

# 7. AGRUPACIÓN POR CIUDAD Y BARRIO
# Le pasamos una lista con las dos columnas ['city', 'neighbourhood_name']
ranking_ml = df_ml.groupby(['city', 'neighbourhood_name'])['Score_Optimizacion'].median().reset_index()

# Ordenamos de mayor a menor potencial
ranking_ml = ranking_ml.sort_values('Score_Optimizacion', ascending=False)

# Vemos el Top 10 con su ciudad al lado
ranking_ml.head(10)

,city,neighbourhood_name,Score_Optimizacion
205,Girona,vilaju�ga,1.157450
435,Sevilla,"prado, parque mar�a luisa",1.123194
471,Valencia,el calvari,1.053196
497,Valencia,la vega baixa,1.038555
170,Girona,sant ferriol,1.036371
483,Valencia,favara,1.023739
405,Sevilla,"cruz roja, capuchinos",1.018808
424,Sevilla,"la palmilla, doctor mara��n",1.016889
41,Barcelona,la trinitat vella,1.007161
65,Barcelona,vallbona,1.006465


In [17]:
# 1. Calculamos las distancias individuales (los "sub-apartados") para saber POR QUÉ fallan
df_ml['dist_rating'] = 100 - df_ml['review_scores_rating']
df_ml['dist_location'] = 100 - df_ml['review_scores_location']
df_ml['dist_min_nights'] = abs(df_ml['minimum_nights'] - 2)
df_ml['dist_max_nights'] = abs(df_ml['maximum_nights'] - 30)

# 2. Elegimos las columnas que queremos ver en la tabla final
columnas_a_mostrar = [
    'dist_rating', 
    'dist_location', 
    'dist_min_nights', 
    'dist_max_nights', 
    'Score_Optimizacion' # Mantenemos la nota de la IA al final
]

# 3. Agrupamos por ciudad y barrio, sacando la mediana de todo
ranking_detallado = df_ml.groupby(['city', 'neighbourhood_name'])[columnas_a_mostrar].median().reset_index()

# 4. ORDENAMOS usando el algoritmo de Machine Learning (Score_Optimizacion)
ranking_detallado = ranking_detallado.sort_values('Score_Optimizacion', ascending=False)

# 5. Parche estético para arreglar los nombres rotos que vimos en tu captura
parches_esteticos = {
    'vilajuga': 'vilajuïga',
    'prado, parque mara luisa': 'prado, parque maría luisa',
    'la palmilla, doctor maran': 'la palmilla, doctor marañón',
    'opael': 'opañel',
    'el calvari': 'el calvari', # Ya está en minúscula, pero por si acaso
    'la vega baixa': 'la vega baixa'
}
ranking_detallado['neighbourhood_name'] = ranking_detallado['neighbourhood_name'].replace(parches_esteticos)

# 6. Vemos el Top 10 con todo el detalle de la IA + las explicaciones
ranking_detallado.head(10)

,city,neighbourhood_name,dist_rating,dist_location,dist_min_nights,dist_max_nights,Score_Optimizacion
205,Girona,vilaju�ga,30.0,40.0,1.0,1095.0,1.157450
435,Sevilla,"prado, parque mar�a luisa",40.0,20.0,0.0,1095.0,1.123194
471,Valencia,el calvari,25.0,20.0,1.0,1095.0,1.053196
497,Valencia,la vega baixa,27.0,10.0,0.0,1095.0,1.038555
170,Girona,sant ferriol,20.0,20.0,0.0,1095.0,1.036371
483,Valencia,favara,16.5,10.0,0.0,1095.0,1.023739
405,Sevilla,"cruz roja, capuchinos",13.0,20.0,0.0,1095.0,1.018808
424,Sevilla,"la palmilla, doctor mara��n",12.0,20.0,0.0,1095.0,1.016889
41,Barcelona,la trinitat vella,0.0,10.0,149.5,1095.0,1.007161
65,Barcelona,vallbona,3.0,20.0,1.0,1095.0,1.006465
